### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
sys.path.append('./utils')

### Random seed for reproducibility

In [2]:
import torch
import random
import numpy as np
#import multiprocessing as mp
#mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [3]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import gc
import svg_constraints 
from svg_processor import SVGSanitizer, SVGProcessor

class Model:
    
    def __init__(self):

        self.model_path="./lora/Qwen25_7B_Instruct_lora_fp16_r256_s2000_i1000_msl2048_awq"
        self.model = LLM(
            model=self.model_path,
            max_model_len=1024,
            #quantization="AWQ",
            gpu_memory_utilization=0.85,
            dtype="half",
            seed=123,
            disable_log_stats=True
        )
       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     


    def _format_prompt(self, description: str) -> str:
        return  f"""Below is an instruction that describes a task, paired with an input that provides further context. 
                Write a response that appropriately completes the request.
                
                ### Instruction:
                Generate a SVG code for the given input:
                
                ### Input:
                {description}
                
                ### Response:
                """
    
    def get_response(self, descriptions):
        
        formatted_input = [self._format_prompt(desc) for desc in descriptions]
        sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=1024,n=1)
        outputs = self.model.generate(formatted_input, sampling_params)
        
        #suitable for batch inputs as well
        output_list=[]
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
            output_list.append(generated_text.strip())
        return output_list
    
    def predict(self, descriptions: list[str], max_new_tokens=1024) -> list[str]:
        output_decoded_list = self.get_response(descriptions)
        final_svg_code_list = []
    
        for description, output in zip(descriptions, output_decoded_list):
            base_svg = SVGProcessor.clean_and_extract_svgs(output, self.default_svg)
            clean_svg = self.sanitizer.enforce_constraints(base_svg)
            final_svg = SVGProcessor.svg_conversion_check(description, clean_svg, self.default_svg)
            final_svg_code_list.append(final_svg)
    
        return final_svg_code_list


INFO 04-30 23:02:09 [__init__.py:239] Automatically detected platform cuda.


In [4]:
model=Model()

WARNING 04-30 23:02:10 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-30 23:02:15 [config.py:585] This model supports multiple tasks: {'embed', 'generate', 'classify', 'reward', 'score'}. Defaulting to 'generate'.
INFO 04-30 23:02:16 [awq_marlin.py:114] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 04-30 23:02:16 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-30 23:02:16 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/Qwen25_7B_Instruct_lora_fp16_r256_s2000_i1000_msl2048_awq', speculative_config=None, tokenizer='./lora/Qwen25_7B_Instruct_lora_fp16_r256_s2000_i1000_msl2048_awq', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_c

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-30 23:02:18 [loader.py:447] Loading weights took 1.37 seconds
INFO 04-30 23:02:19 [gpu_model_runner.py:1186] Model loading took 5.2048 GB and 1.789864 seconds
INFO 04-30 23:02:26 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/ffcaed2437/rank_0_0 for vLLM's torch.compile
INFO 04-30 23:02:26 [backends.py:425] Dynamo bytecode transform time: 7.48 s
INFO 04-30 23:02:27 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 04-30 23:02:31 [monitor.py:33] torch.compile takes 7.48 s in total
INFO 04-30 23:02:33 [kv_cache_utils.py:566] GPU KV cache size: 33,584 tokens
INFO 04-30 23:02:33 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 32.80x
INFO 04-30 23:02:50 [gpu_model_runner.py:1534] Graph capturing finished in 17 secs, took 0.59 GiB
INFO 04-30 23:02:50 [core.py:151] init engine (profile, create kv cache, warmup model) took 31.66 seconds


In [5]:
#model.predict(['who are you?'])

In [6]:
import pandas as pd
df1=pd.read_csv('./drawing-with-llms/svg_test_1_vqa.csv',header=[0])
df2=pd.read_csv('./drawing-with-llms/svg_test_2_vqa_kaggle.csv',header=[0])
df2=df2.drop_duplicates(['description'])
#df=pd.concat([df1[['description']],df2['description']],axis=0)
df=df2
print(df.shape)
df.head(2)

(99, 5)


,id,description,question,choices,answer
0,75c6e0,a vibrant orange sunset over a calm ocean,What is the main subject of the image?,"['mountain', 'ocean', 'river', 'sky']",ocean
4,d060e9,a fluffy white sheep in a green meadow,What color is the sheep?,"['black', 'brown', 'green', 'white']",white


In [7]:
# from tqdm import tqdm
# tqdm.pandas()
# df['svg_3'] = df['description'].progress_apply(lambda x: model.predict(x))

In [8]:
from tqdm import tqdm
description_list = [s.strip(" ',") for s in df['description'].to_list()]
batch_size = 6
results = []

for i in tqdm(range(0, len(description_list), batch_size), desc="Batch prediction"):
    batch = description_list[i:i + batch_size]
    batch_result = model.predict(batch)  # Ensure this handles a list of inputs
    results.extend(batch_result)


Batch prediction:   0%|                                  | 0/17 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/6 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
cessed prompts:  17%|▏| 1/6 [00:04<00:24,  4.93s/it, est. speed input: 12.99 
cessed prompts:  50%|▌| 3/6 [00:06<00:05,  1.75s/it, est. speed input: 30.57 
cessed prompts:  67%|▋| 4/6 [00:06<00:02,  1.25s/it, est. speed input: 39.89 
cessed prompts:  83%|▊| 5/6 [00:07<00:01,  1.30s/it, est. speed input: 41.25 
Processed prompts: 100%|█| 6/6 [00:09<00:00,  1.61s/it, est. speed input: 40.46 
ERROR:root:SVG Parse Error: Unescaped '<' not allowed in attributes values, line 1, column 2126 (<string>, line 1). Returning default SVG.
Batch prediction:   6%|█▌                        | 1/17 [00:09<02:37,  9.84s/it]
cessed prompts:   0%| | 0/6 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
cessed prompts:  17%|▏| 1/6 [00:06<00:30,  6.13s/it, est. speed input: 10.11 
cessed prompts:  33%|▎| 2/6 [00:08<00:16,  4.02s/it, est. speed input: 1

In [9]:
df['svg_3']=results

In [10]:
model.close_model()

In [11]:
from siglip_class import SVGMetricEvaluator
from aesthetic_evaluator import AestheticEvaluator

In [12]:
#SigLip Score
from tqdm import tqdm
tqdm.pandas()
evaluator = SVGMetricEvaluator()
df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)

100%|███████████████████████████████████████████| 99/99 [00:06<00:00, 15.12it/s]


In [13]:
#Aes Score
from tqdm import tqdm
tqdm.pandas()
aes_eval = AestheticEvaluator()
df['aes_score_3'] = df.progress_apply(lambda row: aes_eval.get_score(row['svg_3']), axis=1)

100%|███████████████████████████████████████████| 99/99 [00:11<00:00,  8.93it/s]


In [14]:
#combined score
df['combined_score_3'] = (df['svg_score_3']+df['svg_score_3']+df['aes_score_3'])/3

In [15]:
print('mean_svg_score:',df['svg_score_3'].mean(),'mean_aes_score:',df['aes_score_3'].mean(),'combined_score:',df['combined_score_3'].mean())

mean_svg_score: 0.20515966718917122 mean_aes_score: 0.4373778463614107 combined_score: 0.28256572691325105


In [16]:
default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
df_default_svg=df[df['svg_3']==default_svg]
print('default_svg_count:',df_default_svg.shape[0])
print('default_svg_score_mean:',df_default_svg['svg_score_3'].mean(),'default_aes_score_mean:',df_default_svg['aes_score_3'].mean(),\
     'combined_score:',df_default_svg['combined_score_3'].mean())

default_svg_count: 20
default_svg_score_mean: 4.745116958708786e-08 default_aes_score_mean: 0.4369946956634522 combined_score: 0.1456649301885971


In [17]:
df_non_default_svg=df[df['svg_3']!=default_svg]
print('non-default_svg_count:',df_non_default_svg.shape[0])
print('non-default_svg_score_mean:',df_non_default_svg['svg_score_3'].mean(),\
      'non-default_aes_score_mean:',df_non_default_svg['aes_score_3'].mean(),\
        'combined_score:',df_non_default_svg['combined_score_3'].mean())

non-default_svg_count: 79
non-default_svg_score_mean: 0.25709881142664 non-default_aes_score_mean: 0.43747484653810903 combined_score: 0.31722415646379637
